# 안전모 + 안전벨트 4클래스 학습 (Colab)
런타임 유형을 **T4 GPU**로 바꾼 뒤 위에서부터 순서대로 실행하세요.

사전 준비: 왼쪽 🔑 **Secrets**에 `ROBOFLOW_API_KEY`를 추가하고 '노트북 액세스'를 켜세요.

In [ ]:
!nvidia-smi | head -15
!pip install -q ultralytics

In [ ]:
!git clone -b feature/ppe-v2-helmet-belt https://github.com/2026-DASOM-EXPO/AI.git /content/AI
%cd /content/AI

### 데이터셋 다운로드 (Roboflow API, Mac을 거치지 않음)

In [ ]:
import io, json, os, urllib.request, zipfile
from google.colab import userdata
key = userdata.get('ROBOFLOW_API_KEY')
def dl(slug, ver, out):
    r = json.load(urllib.request.urlopen(f'https://api.roboflow.com/{slug}/{ver}/yolov8?api_key={key}'))
    z = urllib.request.urlopen(r['export']['link']).read()
    zipfile.ZipFile(io.BytesIO(z)).extractall(out); print(out, len(z)//2**20, 'MB')
dl('proyecto-prevencion-predictiva/work-at-height-safety', 1, 'datasets/external/work_at_height')  # 벨트(harness) + helmet
dl('aaa-jmuwe/safe-om9dk', 1, 'datasets/external/safe_helmet_vest')  # helmet 착용/미착용

In [ ]:
!python scripts/prepare_ppe_v2.py --no-base \
  --source datasets/external/work_at_height:wah:0=3,1=1 \
  --source datasets/external/safe_helmet_vest:safe:2=0,4=1

### 학습 (T4 기준 수 시간. 끊김 대비로 Drive에 저장)
먼저 `--epochs 3`으로 짧게 돌려 동작을 확인한 뒤, 정상이면 100으로 올리세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!python scripts/train_custom.py --data datasets/data_ppe_v2.yaml --name ppe_v2 --epochs 3

In [ ]:
# 본 학습 (위 확인 후). project를 Drive로 두면 세션이 끊겨도 결과가 남습니다.
from ultralytics import YOLO
YOLO('models/yolov8n.pt').train(data='datasets/data_ppe_v2.yaml', epochs=100, imgsz=640, batch=-1,
    device=0, project='/content/drive/MyDrive/EXPO_runs', name='ppe_v2', patience=20)

In [ ]:
# 결과 확인 + ONNX 변환 (Jetson용). best.pt / best.onnx 를 다운로드하세요.
from ultralytics import YOLO
p = '/content/drive/MyDrive/EXPO_runs/ppe_v2/weights/best.pt'
m = YOLO(p); print(m.val(data='datasets/data_ppe_v2.yaml', split='test').box.maps)
m.export(format='onnx', imgsz=640, opset=12, simplify=True)